# 01 - Data Exploration

This notebook explores the datasets used for Indian monsoon onset benchmarking:
- IMD gridded rainfall (1901-2024)
- ERA5 reanalysis for circulation indices
- Region definitions (Core Monsoon Zone, India bounds)

Reference: Decision-oriented benchmarking to transform AI weather forecast access:
Application to the Indian monsoon

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Import monsoon benchmark modules
from monsoon_benchmark.data import IMDDataLoader, ERA5DataLoader
from monsoon_benchmark.data.regions import (
    CMZ_BOUNDS, INDIA_BOUNDS, WYI_BOUNDS,
    create_benchmark_grid, mask_region
)
from monsoon_benchmark.data.regridding import regrid_to_benchmark

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Region Definitions

The benchmark uses a 4 degree x 4 degree grid over India. The Core Monsoon Zone (CMZ)
is particularly important for assessing monsoon onset skill.

In [ ]:
# Display region bounds
print("India Bounds:")
print(f"  Lat: {INDIA_BOUNDS.lat_min}N - {INDIA_BOUNDS.lat_max}N")
print(f"  Lon: {INDIA_BOUNDS.lon_min}E - {INDIA_BOUNDS.lon_max}E")

print("\nCore Monsoon Zone (CMZ):")
print(f"  Lat: {CMZ_BOUNDS.lat_min}N - {CMZ_BOUNDS.lat_max}N")
print(f"  Lon: {CMZ_BOUNDS.lon_min}E - {CMZ_BOUNDS.lon_max}E")

print("\nWebster-Yang Index Region:")
print(f"  Lat: {WYI_BOUNDS.lat_min}N - {WYI_BOUNDS.lat_max}N")
print(f"  Lon: {WYI_BOUNDS.lon_min}E - {WYI_BOUNDS.lon_max}E")

In [ ]:
# Create benchmark grid
grid = create_benchmark_grid(resolution=4.0)
print(f"\nBenchmark grid:")
print(f"  Latitudes: {grid['lat'].values}")
print(f"  Longitudes: {grid['lon'].values}")
print(f"  Total grid cells: {len(grid['lat']) * len(grid['lon'])}")

In [ ]:
# Visualize the grid and regions
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    
    fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.PlateCarree()})
    
    ax.coastlines(resolution='50m')
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.add_feature(cfeature.LAND, alpha=0.3)
    
    # Plot grid cells
    for lat in grid['lat'].values:
        for lon in grid['lon'].values:
            rect = plt.Rectangle(
                (lon - 2, lat - 2), 4, 4,
                fill=False, edgecolor='blue', alpha=0.5,
                transform=ccrs.PlateCarree()
            )
            ax.add_patch(rect)
    
    # Highlight CMZ
    cmz_rect = plt.Rectangle(
        (CMZ_BOUNDS.lon_min, CMZ_BOUNDS.lat_min),
        CMZ_BOUNDS.lon_max - CMZ_BOUNDS.lon_min,
        CMZ_BOUNDS.lat_max - CMZ_BOUNDS.lat_min,
        fill=True, facecolor='red', alpha=0.3, edgecolor='red', linewidth=2,
        transform=ccrs.PlateCarree(), label='CMZ'
    )
    ax.add_patch(cmz_rect)
    
    ax.set_extent([60, 100, 5, 40], crs=ccrs.PlateCarree())
    ax.gridlines(draw_labels=True, alpha=0.3)
    ax.set_title('4x4 Degree Benchmark Grid over India')
    ax.legend(loc='upper left')
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("Cartopy not installed. Skipping map visualization.")
    print("Install with: pip install cartopy")

## 2. IMD Rainfall Data

The India Meteorological Department (IMD) provides 1 degree gridded daily rainfall
data from 1901 to present. This is the primary observational dataset for
defining monsoon onset.

In [ ]:
# Configure data path (adjust as needed)
DATA_DIR = Path('../data')
IMD_FILE = DATA_DIR / 'imd_rainfall_1901_2024.nc'

# Check if data exists
if IMD_FILE.exists():
    print(f"IMD data found at: {IMD_FILE}")
else:
    print(f"IMD data not found at: {IMD_FILE}")
    print("\nTo download IMD data, use the IMDDataLoader:")
    print("  loader = IMDDataLoader(data_dir='../data')")
    print("  loader.download_imd_rainfall(start_year=1901, end_year=2024)")

In [ ]:
# Load IMD data if available (or create synthetic data for demonstration)
try:
    loader = IMDDataLoader(data_dir=DATA_DIR)
    imd_data = loader.load_rainfall()
    print("Loaded IMD rainfall data:")
    print(imd_data)
except FileNotFoundError:
    print("Creating synthetic data for demonstration...")
    
    # Create synthetic rainfall data
    from monsoon_benchmark.data.imd import create_synthetic_imd_data
    imd_data = create_synthetic_imd_data(years=range(2019, 2025))
    print("Created synthetic IMD data:")
    print(imd_data)

In [ ]:
# Examine rainfall climatology
if 'rainfall' in imd_data:
    # Compute daily climatology
    clim = imd_data['rainfall'].groupby('time.dayofyear').mean()
    
    # CMZ average
    cmz_clim = clim.sel(
        lat=slice(CMZ_BOUNDS.lat_min, CMZ_BOUNDS.lat_max),
        lon=slice(CMZ_BOUNDS.lon_min, CMZ_BOUNDS.lon_max)
    ).mean(dim=['lat', 'lon'])
    
    # Plot seasonal cycle
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(cmz_clim.dayofyear, cmz_clim.values, 'b-', linewidth=2)
    ax.axvline(152, color='red', linestyle='--', label='June 1 (DOY 152)')
    ax.axvline(180, color='orange', linestyle='--', label='June 29 (DOY 180)')
    ax.set_xlabel('Day of Year')
    ax.set_ylabel('Rainfall (mm/day)')
    ax.set_title('CMZ Daily Rainfall Climatology')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(90, 270)  # April to September
    plt.tight_layout()
    plt.show()

## 3. Regridding to 4x4 Degree Grid

The benchmark evaluates models on a coarse 4 degree x 4 degree grid to:
1. Match the scale of actionable agricultural decisions
2. Reduce noise in onset detection
3. Enable fair comparison across models with different native resolutions

In [ ]:
# Regrid to benchmark grid
try:
    regridded = regrid_to_benchmark(
        imd_data['rainfall'],
        target_resolution=4.0,
        method='conservative'
    )
    print("Regridded data:")
    print(regridded)
except ImportError:
    print("xesmf not installed. Using simple averaging for demonstration.")
    print("Install with: pip install xesmf")
    
    # Simple coarsening as fallback
    regridded = imd_data['rainfall'].coarsen(
        lat=4, lon=4, boundary='trim'
    ).mean()
    print("\nCoarsened data:")
    print(regridded)

In [ ]:
# Compare original and regridded data for a sample day
if 'rainfall' in imd_data:
    sample_time = imd_data.time.values[150]  # Mid-June
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Original resolution
    ax1 = axes[0]
    imd_data['rainfall'].sel(time=sample_time).plot(
        ax=ax1, cmap='Blues', vmin=0, vmax=30
    )
    ax1.set_title('Original IMD (1 degree)')
    
    # Regridded
    ax2 = axes[1]
    regridded.sel(time=sample_time, method='nearest').plot(
        ax=ax2, cmap='Blues', vmin=0, vmax=30
    )
    ax2.set_title('Regridded (4 degree)')
    
    plt.tight_layout()
    plt.show()

## 4. ERA5 Data for Circulation Indices

ERA5 provides atmospheric reanalysis data needed for:
- Webster-Yang Index (WYI): measures monsoon circulation strength
- Initial conditions for AIWP models

In [ ]:
# ERA5 data path
ERA5_FILE = DATA_DIR / 'era5_winds_1965_2024.nc'

if ERA5_FILE.exists():
    print(f"ERA5 data found at: {ERA5_FILE}")
else:
    print(f"ERA5 data not found at: {ERA5_FILE}")
    print("\nTo download ERA5 data, use the ERA5DataLoader:")
    print("  loader = ERA5DataLoader(data_dir='../data')")
    print("  loader.download_era5_winds(start_year=1965, end_year=2024)")

In [ ]:
# Load ERA5 data if available (or create synthetic data)
try:
    era5_loader = ERA5DataLoader(data_dir=DATA_DIR)
    u200 = era5_loader.load_u_wind(level=200)
    u850 = era5_loader.load_u_wind(level=850)
    print("Loaded ERA5 wind data")
except FileNotFoundError:
    print("Creating synthetic ERA5 data for demonstration...")
    from monsoon_benchmark.data.era5 import create_synthetic_era5_data
    era5_data = create_synthetic_era5_data(years=range(2019, 2025))
    u200 = era5_data['u200']
    u850 = era5_data['u850']
    print("Created synthetic ERA5 data")

In [ ]:
# Compute and visualize Webster-Yang Index
from monsoon_benchmark.indices import compute_wyi

# Compute WYI
wyi = compute_wyi(u200, u850)
print(f"Webster-Yang Index shape: {wyi.shape}")
print(f"  Time range: {wyi.time.values[0]} to {wyi.time.values[-1]}")

In [ ]:
# Plot WYI seasonal cycle
wyi_clim = wyi.groupby('time.dayofyear').mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(wyi_clim.dayofyear, wyi_clim.values, 'b-', linewidth=2)
ax.axhline(0, color='gray', linestyle='-', alpha=0.5)
ax.axvline(152, color='red', linestyle='--', label='June 1')
ax.fill_between(
    wyi_clim.dayofyear,
    wyi_clim.values,
    where=wyi_clim.values > 0,
    alpha=0.3,
    color='red',
    label='Monsoon mode (WYI > 0)'
)
ax.set_xlabel('Day of Year')
ax.set_ylabel('WYI (m/s)')
ax.set_title('Webster-Yang Index Seasonal Cycle')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(90, 300)
plt.tight_layout()
plt.show()

## 5. Summary Statistics

Key statistics for the benchmark datasets.

In [ ]:
# Summary statistics
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

if 'rainfall' in imd_data:
    print("\nIMD Rainfall:")
    print(f"  Time range: {pd.to_datetime(imd_data.time.values[0]).strftime('%Y-%m-%d')} to "
          f"{pd.to_datetime(imd_data.time.values[-1]).strftime('%Y-%m-%d')}")
    print(f"  Grid: {len(imd_data.lat)}x{len(imd_data.lon)} "
          f"({float(imd_data.lat[1] - imd_data.lat[0]):.1f} degree)")
    print(f"  Total timesteps: {len(imd_data.time)}")

print("\nERA5 Winds:")
print(f"  U200 shape: {u200.shape}")
print(f"  U850 shape: {u850.shape}")

print("\nBenchmark Grid:")
print(f"  Resolution: 4 degrees")
print(f"  Cells over India: {len(grid['lat']) * len(grid['lon'])}")
print(f"  CMZ cells: ~6-8 cells (18-28N, 74-86E)")

## Next Steps

Continue to:
- **02_onset_index_validation.ipynb**: Compute and validate monsoon onset indices
- **03_deterministic_evaluation.ipynb**: Evaluate deterministic forecasts
- **04_probabilistic_evaluation.ipynb**: Evaluate ensemble forecasts